In [32]:
import pandas as pd
import requests
import urllib.request
import re


In [33]:
# grab url from state election results website

url = "https://electionresultsfiles.sos.mn.gov/20260811/cntyRaces.txt"

In [ ]:
# read in data in .txt format

response = requests.get(url, timeout=30)
response.raise_for_status()
data = response.text
data = data.replace('\r', '')
data = data.split('\n')
# save as .txt format
with open('data/results.txt', 'w') as f:
    for line in data:
        f.write(line)
        f.write('\n')

In [35]:
df = pd.read_csv("data/results.txt",delimiter = ';',header=None,)


df.head(2)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,MN,1,NaN,391,County Commissioner District 1,1.0,9001,Timothy Catlin,NaN,NaN,NP,0,2,0,0.0,0
1,MN,1,NaN,391,County Commissioner District 1,1.0,9002,Christopher S. Dotzler,NaN,NaN,NP,0,2,0,0.0,0


In [43]:
df.columns = [
    'State',
	'County ID (if applicable)',
	'Precinct name (if applicable)',
	'Office ID',
	'Office Name', 
	'District*',
	'Candidate Order Code',
	'Candidate Name (First/Last/Suffix all in one field)',
	'Suffix (not used)',
	'Incumbent Code (not used)',
	'Party Abbreviation',
	'Number of Precincts reporting',
	'Total number of precincts voting for the office',
	'Votes for Candidate',
	'Percentage of Votes for Candidate out of Total Votes for Office',
	'Total number of votes for Office in area'
]    
 
# rename columns
new_col_names = {
    'Candidate Name (First/Last/Suffix all in one field)': 'Candidate',
    'Number of Precincts reporting': 'precincts_reporting',
    'Total number of precincts voting for the office': 'total_precincts',
    'Votes for Candidate': 'Votes',
    'Percentage of Votes for Candidate out of Total Votes for Office': 'Percentage',
    'Total number of votes for Office in area': 'total_votes'
}

In [50]:
    # select columns to keep
cols_to_keep = [
    'Office ID',
    'Office Name', 
    'Candidate Name (First/Last/Suffix all in one field)',
    'Number of Precincts reporting',
    'Total number of precincts voting for the office',
    'Votes for Candidate',
    'Percentage of Votes for Candidate out of Total Votes for Office',
    'Total number of votes for Office in area'
]

In [51]:
def clean_up_df(df):
    
    df['Candidate Name (First/Last/Suffix all in one field)'] = df['Candidate Name (First/Last/Suffix all in one field)'].str.replace("WRITE-IN", "Write-in")
    df = df[cols_to_keep]
    df = df.rename(columns=new_col_names)
    return df


In [52]:
narrow_col = [
    'Candidate',
    'Percentage',
    'Votes'
]

def narrow_df(df):
    df = df[narrow_col]
    return df

In [53]:
def calculate_precincts_reporting(df,x,y):
    pct_precinct = (df['precincts_reporting'].iloc[x]/ df['total_precincts'].iloc[x]) * 100
    pct_precinct = round(pct_precinct,2)
    pct_precinct = "{:g}".format(pct_precinct)
    new_row_values = {'Candidate': str(pct_precinct) + '% of precincts reporting'}
    df.loc[y] = new_row_values   # adding a row
    return df

In [59]:
df.head(2)

,State,County ID (if applicable),Precinct name (if applicable),Office ID,Office Name,District*,Candidate Order Code,Candidate Name (First/Last/Suffix all in one field),Suffix (not used),Incumbent Code (not used),Party Abbreviation,Number of Precincts reporting,Total number of precincts voting for the office,Votes for Candidate,Percentage of Votes for Candidate out of Total Votes for Office,Total number of votes for Office in area
0,MN,1,NaN,391,County Commissioner District 1,1.0,9001,Timothy Catlin,NaN,NaN,NP,0,2,0,0.0,0
1,MN,1,NaN,391,County Commissioner District 1,1.0,9002,Christopher S. Dotzler,NaN,NaN,NP,0,2,0,0.0,0


### Get Hennepin County results

In [62]:
# grab minneapolis mayoral race
attorney = df[(df['Office ID'] == 405)]
attorney = clean_up_df(attorney)
attorney

/var/folders/5t/wxffb02s329cdc135wwp_v100000gn/T/ipykernel_4681/1595333411.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Candidate Name (First/Last/Suffix all in one field)'] = df['Candidate Name (First/Last/Suffix all in one field)'].str.replace("WRITE-IN", "Write-in")


,Office ID,Office Name,Candidate,precincts_reporting,total_precincts,Votes,Percentage,total_votes
70,405,County Attorney,Matt Pelikan,0,395,0,0.0,0
71,405,County Attorney,Diane M. Krenz,0,395,0,0.0,0
72,405,County Attorney,Cedrick Frazier,0,395,0,0.0,0
73,405,County Attorney,Anders Folk,0,395,0,0.0,0
74,405,County Attorney,Hao Nguyen,0,395,0,0.0,0
154,405,County Attorney,Ron Hocevar,0,54,0,0.0,0
155,405,County Attorney,Todd Zettler,0,54,0,0.0,0
156,405,County Attorney,Allen Andersen,0,54,0,0.0,0


In [63]:
attorney = calculate_precincts_reporting(attorney,0,0)

In [64]:
attorney = narrow_df(attorney)
attorney

,Candidate,Percentage,Votes
70,Matt Pelikan,0.0,0.0
71,Diane M. Krenz,0.0,0.0
72,Cedrick Frazier,0.0,0.0
73,Anders Folk,0.0,0.0
74,Hao Nguyen,0.0,0.0
154,Ron Hocevar,0.0,0.0
155,Todd Zettler,0.0,0.0
156,Allen Andersen,0.0,0.0
0,0% of precincts reporting,NaN,NaN


### Save all into csv

In [65]:
# save as csv
attorney.to_csv('output/hennepin_attorney_2026.csv', index=False)
